# Reusable template — logistic GD + sklearn classifier

**Short name:** `LogReg_GD_Sklearn`  
Copy this notebook when you have a binary target and a numeric feature matrix and you want both a from-scratch GD baseline and a scikit-learn fit.

Replace the `DATA_PATH` / column names, then run top to bottom.


In [ ]:
import copy, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

DATA_PATH = "data/logreg_gd_2d.csv"
FEATURE_COLS = ["x0", "x1"]
TARGET_COL = "y"
ALPHA = 0.1
ITERS = 5000
THRESHOLD = 0.5

df = pd.read_csv(DATA_PATH)
X = df[FEATURE_COLS].to_numpy(dtype=float)
y = df[TARGET_COL].to_numpy(dtype=float)
print(X.shape, y.mean())


In [ ]:
def sigmoid(z):
    z = np.clip(np.asarray(z, dtype=float), -50.0, 50.0)
    return 1.0 / (1.0 + np.exp(-z))

def cost(X, y, w, b):
    f = np.clip(sigmoid(X @ w + b), 1e-15, 1 - 1e-15)
    return float(-np.mean(y * np.log(f) + (1 - y) * np.log(1 - f)))

def grad(X, y, w, b):
    err = sigmoid(X @ w + b) - y
    return float(np.mean(err)), (X.T @ err) / X.shape[0]

def fit_gd(X, y, alpha=ALPHA, iters=ITERS):
    w = np.zeros(X.shape[1]); b = 0.0; hist = []
    for i in range(iters):
        dj_db, dj_dw = grad(X, y, w, b)
        w = w - alpha * dj_dw; b = b - alpha * dj_db
        if i % max(iters // 10, 1) == 0:
            hist.append((i, cost(X, y, w, b)))
            print(f"iter {i:5d}  J={hist[-1][1]:.6f}")
    return w, b, hist

w, b, hist = fit_gd(X, y)
proba = sigmoid(X @ w + b)
pred = (proba >= THRESHOLD).astype(int)
print("GD w,b:", w, b, "acc:", np.mean(pred == y), "J:", cost(X, y, w, b))

clf = LogisticRegression(C=np.inf, solver="lbfgs", max_iter=2000)
clf.fit(X, y)
print("sklearn C=inf acc:", clf.score(X, y), "coef:", clf.coef_, "intercept:", clf.intercept_)
clf_l2 = LogisticRegression()
clf_l2.fit(X, y)
print("sklearn L2 acc:", clf_l2.score(X, y), "coef:", clf_l2.coef_)


Swap `DATA_PATH` for `data/logreg_gd_practice.csv` or `data/logreg_gd_dti.csv` (set `FEATURE_COLS=['dti']`).  
Raise `THRESHOLD` to make a more conservative classifier. Lower `ALPHA` if `J` increases.
